# 04e — EXP_05: RAGAS Evaluation (Claude Sonnet 4.6 judge — all 5 metrics)

Scores `results/exp_05_multi_hop_rag__golden_234/predictions.jsonl` against the 234-row golden subset.

**Headline comparison numbers** (Phase 4 final cross-architecture):

| Architecture | Acuuracy (test_1273) | Faithfulness | Hallucination Rate | Context Precision | Context Recall |
|---|---:|---:|---:|---:|---:|
| EXP_01 No-RAG | 0.7738 | n/a | n/a | n/a | n/a |
| EXP_02 Naive Dense | 0.7573 | 0.1308 | 0.8957 | 0.3285 | 0.4124 |
| EXP_03 Sparse BM25 | 0.7581 | (pending) | (pending) | (pending) | (pending) |
| EXP_04 Hybrid | (pending) | (pending) | (pending) | (pending) | (pending) |
| **EXP_05 Multi-Hop** | **??** | **??** | **??** | **??** | **??** |

**Falsifiable hypothesis** (anchored in [`docs/output_notes/04b_exp02_output.md` §3.5](../docs/output_notes/04b_exp02_output.md)): *Multi-Hop achieves Faithfulness > 0 on the 13 multi-hop golden rows where Naive scored 0.000. Falsified if Multi-Hop Faithfulness on `requires_multihop=yes` rows ≤ 0.05.*

Stages: smoke 10 (~$0.50, ~1 min) → full 234 (~$11–12, ~10–20 min) → NaN rescore if needed.

Note: EXP_05 retrieval returns up to 15 chunks per question (vs 5 for EXP_02–04), so RAGAS prompts are longer. Cost may be slightly higher than EXP_02–04 (~$13–15) due to the bigger context.

## 1. Setup

In [ ]:
import json
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from dotenv import load_dotenv

load_dotenv(REPO_ROOT / ".env")

from src.eval.ragas_eval import score_predictions

for k in ["ANTHROPIC_API_KEY"]:
    print(f"{k}: {'\u2713 present' if os.environ.get(k) else '\u2717 MISSING'}")
print("Repo root:", REPO_ROOT)

## 2. Configuration

In [ ]:
PRED_DIR = REPO_ROOT / "results" / "exp_05_multi_hop_rag__golden_234"
GOLDEN_PATH = REPO_ROOT / "data" / "processed" / "golden_ragas_300.jsonl"
CHUNKS_PATH = REPO_ROOT / "data" / "processed" / "chunks.parquet"

RUN_SMOKE = True
SMOKE_N = 10
RUN_FULL = False
RUN_RESCORE_NANS = False
OVERWRITE_SCORES = False

print("Predictions:", PRED_DIR)
print("Golden     :", GOLDEN_PATH)
print("Chunks     :", CHUNKS_PATH)

assert PRED_DIR.exists(), (
    f"{PRED_DIR} doesn't exist — run 04e_exp05_multi_hop_rag.ipynb Stage B first."
)

## 3. Stage A — Smoke (10 rows × 5 metrics · ~1 min · ~$0.50–0.80)

In [ ]:
import pandas as pd
import shutil

if RUN_SMOKE:
    smoke_dir = PRED_DIR.parent / "exp_05_multi_hop_rag__golden_234_ragas_smoke"
    smoke_dir.mkdir(parents=True, exist_ok=True)
    for fname in ("predictions.jsonl", "retrieval.jsonl", "summary.json"):
        shutil.copy2(PRED_DIR / fname, smoke_dir / fname)

    smoke_summary = score_predictions(
        predictions_dir=smoke_dir,
        golden_path=GOLDEN_PATH,
        chunks_path=CHUNKS_PATH,
        n_smoke=SMOKE_N,
        overwrite=OVERWRITE_SCORES,
    )
    print('\n=== Stage A smoke summary (10 rows × 5 metrics) ===')
    for k, v in smoke_summary.items():
        if k.startswith(("RAGAS_", "Answer_", "ragas_")):
            print(f"  {k:30s} : {v}")
    smoke_scores = pd.read_csv(smoke_dir / "ragas_scores.csv")
    cols = [c for c in smoke_scores.columns if c in ("question_id", "_is_correct", "faithfulness", "context_precision", "context_recall", "answer_relevancy", "answer_correctness")]
    print("\n=== Per-row scores (smoke 10) ===")
    print(smoke_scores[cols].to_string(index=False))
else:
    print("RUN_SMOKE = False — skipped.")

**Acceptance gates (Stage A)** — same as EXP_04: no errors, all 5 metric cols present, NaN < 30 %, Faithfulness ≥ 0 on at least some rows, AC correlates with `_is_correct`.

## 4. Stage B — Full 234 (~10–20 min · ~$13–15)

Multi-Hop's larger context (up to 15 chunks vs 5) makes RAGAS prompts longer than EXP_02–04. Cost projection is slightly higher; user can also stop after Stage A if budget tight.

In [ ]:
if RUN_FULL:
    full_summary = score_predictions(
        predictions_dir=PRED_DIR,
        golden_path=GOLDEN_PATH,
        chunks_path=CHUNKS_PATH,
        n_smoke=None,
        overwrite=OVERWRITE_SCORES,
    )
    print("\n=== Stage B full summary (234 rows × 5 metrics) ===")
    for k, v in full_summary.items():
        if k.startswith(("RAGAS_", "Answer_", "ragas_")) or k == "Acuuracy":
            print(f"  {k:30s} : {v}")
else:
    print("RUN_FULL = False")

## 5. Stage C — NaN rescore

In [ ]:
if RUN_RESCORE_NANS:
    rescored_summary = score_predictions(
        predictions_dir=PRED_DIR,
        golden_path=GOLDEN_PATH,
        chunks_path=CHUNKS_PATH,
        rescore_nans=True,
    )
    df = pd.read_csv(PRED_DIR / "ragas_scores.csv")
    for col in ("faithfulness", "context_precision", "context_recall", "answer_relevancy", "answer_correctness"):
        if col in df.columns:
            n_nan = pd.to_numeric(df[col], errors="coerce").isna().sum()
            print(f"  {col:25s}: {n_nan} NaN remaining of {len(df)}")
else:
    print("RUN_RESCORE_NANS = False")

## 6. Inspect — does Multi-Hop achieve Faithfulness > 0 on multi-hop rows?

In [ ]:
scores_path = PRED_DIR / "ragas_scores.csv"
if scores_path.exists():
    df = pd.read_csv(scores_path)
    print(f"\nrows: {len(df)}")

    # Cross-architecture overall comparison (golden 234)
    print('\n--- EXP_05 vs other architectures (golden 234) ---')
    e2 = json.loads((REPO_ROOT / 'results' / 'exp_02_naive_rag__golden_234' / 'summary.json').read_text())
    for col, key in [('faithfulness', 'RAGAS_Faithfulness'), ('context_precision', 'RAGAS_Context_Precision'), ('context_recall', 'RAGAS_Context_Recall'), ('answer_relevancy', 'RAGAS_Answer_Relevance'), ('answer_correctness', 'Answer_Correctness')]:
        if col not in df.columns: continue
        s = pd.to_numeric(df[col], errors='coerce')
        e5_mean = s.mean()
        e2_mean = e2.get(key)
        delta_s = f'{(e5_mean - e2_mean)*100:+.2f} pp' if isinstance(e2_mean, (int, float)) else 'n/a'
        print(f'  {col:25s}: EXP_05={e5_mean:.4f}  EXP_02={e2_mean if e2_mean else "--":>8}  Δ={delta_s}  NaN={s.isna().sum()}')

    # Multi-hop-specific subset — where the falsifiable hypothesis is tested
    golden_rows = [json.loads(l) for l in open(REPO_ROOT / 'data' / 'processed' / 'golden_ragas_300.jsonl')]
    mh_qids = {f"golden_{r['question_id']:03d}" for r in golden_rows if r.get('requires_multihop') == 'yes'}
    mh_df = df[df['question_id'].isin(mh_qids)]
    print(f'\n  Multi-hop subset (requires_multihop=yes): n={len(mh_df)}')
    for col in ('faithfulness', 'context_precision', 'context_recall', 'answer_correctness'):
        if col in mh_df.columns:
            s = pd.to_numeric(mh_df[col], errors='coerce')
            print(f'    {col:25s}: mean={s.mean():.4f}  (EXP_02 had F=0.000 on these rows)')
    if 'faithfulness' in mh_df.columns:
        f_mh = pd.to_numeric(mh_df['faithfulness'], errors='coerce').mean()
        verdict = '\u2713 SUPPORTED' if f_mh > 0.05 else '\u2717 FALSIFIED'
        print(f'\n  Hypothesis: Multi-Hop Faithfulness > 0.05 on multi-hop rows? → {verdict}  (got {f_mh:.4f})')
else:
    print("No ragas_scores.csv yet — run Stage B first.")

---

**Next.** With all 5 baseline + 5 RAGAS evaluations done, write the **cross-architecture analysis** for Phase 4 — Table 1 of the thesis. The 4-RAG matrix (Naive · Sparse · Hybrid · Multi-Hop) is now fully populated, ready for the discussion-chapter narrative.